In [1]:
#importing the Libraries
import pandas as pd
# Reading the Dataset
dataset = pd.read_csv('Social_Network_Ads.csv')
dataset=pd.get_dummies(dataset,drop_first=True).astype(int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [2]:
independent=dataset[['Age', 'EstimatedSalary','Gender_Male']]
dependent=dataset['Purchased']

In [4]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(independent, dependent, test_size = 0.20, random_state = 0)

In [5]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [8]:
from sklearn.tree import DecisionTreeClassifier #https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

from sklearn.model_selection import GridSearchCV
param_grid = {'criterion':['gini', 'entropy', 'log_loss'],
'max_features': ['sqrt','log2'],
'splitter':['best','random']}
grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train, Y_train) 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'max_features': ['sqrt', 'log2'], 'splitter': ['best', 'random']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [11]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test) 

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test, grid_predictions)
print(cm)
# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(Y_test, grid_predictions)
print(clf_report)

[[54  4]
 [ 2 20]]
              precision    recall  f1-score   support

           0       0.96      0.93      0.95        58
           1       0.83      0.91      0.87        22

    accuracy                           0.93        80
   macro avg       0.90      0.92      0.91        80
weighted avg       0.93      0.93      0.93        80



In [13]:
from sklearn.metrics import f1_score
f1_macro=f1_score(Y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'criterion': 'gini', 'max_features': 'sqrt', 'splitter': 'best'}: 0.9259725400457665


In [16]:
from sklearn.metrics import roc_auc_score
roc_auc_score(Y_test,grid.predict_proba(X_test)[:,1])

0.9200626959247649

In [17]:
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004263,0.000790,0.010243,0.001545,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.829552,0.808442,0.843750,0.846096,0.936903,0.852949,0.044064,1
1,0.002677,0.001813,0.005618,0.000609,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.752503,0.793775,0.747614,0.858747,0.921526,0.814833,0.066571,10
2,0.001735,0.000333,0.003972,0.000598,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.859841,0.776515,0.829832,0.876518,0.857229,0.839987,0.035094,4
3,0.002142,0.000735,0.008940,0.004022,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.813639,0.764578,0.843750,0.779162,0.793775,0.798981,0.027650,11
4,0.003992,0.000690,0.012798,0.000717,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.860542,0.825502,0.844872,0.846096,0.840368,0.843476,0.011250,3
5,0.005798,0.001212,0.015152,0.002592,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.764717,0.758892,0.861321,0.812500,0.905355,0.820557,0.056282,7
6,0.006812,0.001334,0.017879,0.001303,entropy,log2,best,"{'criterion': 'entropy', 'max_features': 'log2...",0.796088,0.840368,0.814777,0.906923,0.872294,0.846090,0.039751,2
7,0.006428,0.000958,0.015875,0.003888,entropy,log2,random,"{'criterion': 'entropy', 'max_features': 'log2...",0.751518,0.791040,0.830822,0.793775,0.813846,0.796200,0.026605,12
8,0.007534,0.001549,0.016114,0.001970,log_loss,sqrt,best,"{'criterion': 'log_loss', 'max_features': 'sqr...",0.798561,0.857229,0.798892,0.845648,0.855335,0.831133,0.026750,5
9,0.006200,0.000805,0.014651,0.002685,log_loss,sqrt,random,"{'criterion': 'log_loss', 'max_features': 'sqr...",0.796088,0.795968,0.798892,0.891711,0.812500,0.819032,0.036848,8
